# Stage C 03k — Medium A100 capacity and numeric qualification


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='4b35abbc6a0970fcbbff47b6b1f814f01e668bf1'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_ROOT='/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/inputs/ecoli_skani_triangle.tsv'
NCBI_ZIP_DIR=f'{SOURCE_ROOT}/raw/ncbi_dataset_zips'


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise
def record_once(run_id,artifact,tier):
    marker=STUDY_ROOT/'record_markers'/f'{run_id}.json'
    if marker.exists():
        print('Ledger record already exists:',marker); return
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier',tier,'--artifact',str(artifact)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')


In [ ]:
import torch
if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name(0).upper(): raise RuntimeError('Select an A100 runtime.')
root=Path(DRIVE_ROOT)/'runs/c18_v3_medium_a100_qualification'
command=['seqtrainer-titans-stage-c-capacity','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'e25.json'),'--validation-panel-manifest',str(panels/'validation.json'),'--output-dir',str(root),'--require','A100','--horizons','3','--variants','exact_sdpa_fp32','exact_sdpa_bfloat16','--steps','10','--batch-size','1','--block-count','12','--d-model','256','--num-heads','8','--persistent-tokens','4','--memory-depth','2','--memory-architecture','paper_residual_mlp_v2','--memory-recurrence-policy','paper_exact','--validation-segments','32']
run_logged(root,'medium_capacity',command)
report=json.loads((root/'capacity_matrix.json').read_text())
fp32=next(x for x in report['results'] if x['variant']=='exact_sdpa_fp32')
total=torch.cuda.get_device_properties(0).total_memory
eligible=[x for x in report['results'] if x['available'] and x['peak_allocated_bytes']<=0.70*total and abs(x['validation_bpb']-fp32['validation_bpb'])<=0.005 and x['finite']]
if not eligible: raise RuntimeError('No qualified activation; do not run 03l.')
best=max(eligible,key=lambda x:x['bases_per_second'])
selection={'format_version':1,'passed':True,'activation':{'exact_sdpa_fp32':'float32','exact_sdpa_bfloat16':'bfloat16'}[best['variant']],'batch_size':1,'selected':best,'total_gpu_bytes':total}
(root/'qualification_selection.json').write_text(json.dumps(selection,indent=2,sort_keys=True)+'\n')
record_once('medium_a100_qualification_v1',root,'engineering')
print(json.dumps(selection,indent=2)); print('SHARE THIS DIRECTORY:',root)
